# Debug: #385 — characters reordered around double line breaks

**Symptom** (from the issue): a lattice cell whose visual text is
```
ABC
DEF

GHI
```
comes out as `GABC\nDEF\nHI` — the first character after the blank line floats to the front.

**Root cause hypothesis:** `_process_horizontal_cut` iterates `textline._objs` in playa's emission order, which isn't reading order around blank lines.

**The speculative fix** (PR #758, closed because it broke `test_lattice_split_text`): sort LTChars by `(-y, x)` and synthesise line breaks on y-jumps. This notebook lets you decide whether the fix is correct (and the lattice fixture needs updating) or over-corrects.

Issue: [#385](https://github.com/camelot-dev/camelot/issues/385) · closed PR: #758

**Setup:** `pip install -e .[plot]`, then run.

## ⚙️ Colab bootstrap (run me first)

On Google Colab this clones the `debug/385-char-reading-order` branch, installs camelot editable, and cd-s in. On a local checkout it is a no-op.


In [ ]:
import sys, os, subprocess

BRANCH = "debug/385-char-reading-order"
REPO = "https://github.com/bosd/camelot.git"

if "google.colab" in sys.modules:
    if os.path.basename(os.getcwd()) != "camelot" and not os.path.isdir("camelot/.git"):
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO], check=True)
    if os.path.basename(os.getcwd()) != "camelot":
        os.chdir("camelot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[plot]", "pytest"], check=True)
    print("Colab bootstrap complete -> cwd:", os.getcwd())
else:
    print("Not on Colab - assuming a local editable checkout; skipping bootstrap.")


In [ ]:
# Fetch the issue's reproducer PDF (1b.pdf).
import urllib.request, os
REPRO_URL = 'https://github.com/camelot-dev/camelot/files/11950066/1b.pdf'
os.makedirs('/tmp/camelot-385', exist_ok=True)
dst = '/tmp/camelot-385/1b.pdf'
if not os.path.exists(dst):
    try:
        urllib.request.urlretrieve(REPRO_URL, dst)
        print('downloaded', dst)
    except Exception as e:
        print('download failed:', e)
        print('Manually download from the issue and save to', dst)
else:
    print('already have', dst)

## What's actually going on (root cause)

The bug is **not** a within-textline character-order problem — each textline is internally fine. Two things combine:

1. **`Cell.text` is an appending setter** (`self._text = ''.join([self._text, t])` in `camelot/core.py`) — every textline assigned to a cell concatenates in *processing order*.
2. **`compute_parse_errors` processes `for direction in ['vertical', 'horizontal']`** — the vertical pass runs first. A lone ambiguous glyph (e.g. the `d` of *dihydroclorid*, which playa split into its own one-char textline `'  d'`) gets classified **vertical** and is appended *before* the horizontal textlines (`Eprazinon`, `ihydrocl`, `orid`).

Result: `'d\nEprazinon \nihydrocl\norid'` — the `d` floats to the front.

### The fix
Process all textlines from both directions in **global reading order `(-y0, x0)`** before the appending setter runs. Then the cell accumulates in reading order: `Eprazinon` → `d` → `ihydrocl` → `orid` = *Eprazinon dihydroclorid*.

## Step 1 — confirm the 5-textline split (the evidence)
Spy on `get_table_index` to see the textlines feeding the scrambled cell.

In [ ]:
import camelot.parsers.base as B
from playa.miner import LTChar

_orig = B.get_table_index
captured = []
def _spy(table, t, direction, split_text=False, flag_size=False, strip_text=''):
    txt = t.get_text()
    if 'prazinon' in txt or 'hydrocl' in txt or txt.strip() == 'd':
        captured.append((direction, repr(txt), round(t.y0,1), round(t.x0,1)))
    return _orig(table, t, direction, split_text, flag_size, strip_text)

B.get_table_index = _spy
import camelot
camelot.read_pdf('/tmp/camelot-385/1b.pdf', flavor='lattice')
B.get_table_index = _orig

for direction, txt, y0, x0 in captured:
    print(f'{direction:10} y0={y0:7} x0={x0:7}  {txt}')
# Watch for a one-char 'd' textline tagged 'vertical' with the HIGHEST-in-list
# position despite a lower y0 than 'Eprazinon' — that's the float.

## Step 2 — apply the fix (global reading-order processing)
Monkey-patch `BaseParser.compute_parse_errors` to flatten both direction passes into one `(-y0, x0)`-sorted stream.

In [ ]:
import camelot.parsers.base as B
from camelot.utils import get_table_index, text_replace

def compute_parse_errors_readingorder(self, table):
    pos_errors = []
    # #385: process textlines from BOTH directions in global reading
    # order (-y0, x0) so the appending Cell.text setter accumulates
    # fragments top-to-bottom, left-to-right -- not vertical-pass-first.
    all_tl = [(t, d) for d in ('vertical', 'horizontal') for t in self.t_bbox[d]]
    all_tl.sort(key=lambda td: (-td[0].y0, td[0].x0))
    for t, direction in all_tl:
        indices, error = get_table_index(
            table, t, direction,
            split_text=self.split_text, flag_size=self.flag_size,
            strip_text=self.strip_text,
        )
        if len(indices) > 0 and indices[0][:2] != (-1, -1):
            pos_errors.append(error)
            indices = type(self)._reduce_index(table, indices, shift_text=self.shift_text)
            for r_idx, c_idx, text in indices:
                if self.replace_text:
                    text = text_replace(text, self.replace_text)
                table.cells[r_idx][c_idx].text = text
    return pos_errors

B.BaseParser.compute_parse_errors = compute_parse_errors_readingorder
print('patched BaseParser.compute_parse_errors')

In [ ]:
import importlib, camelot
importlib.reload(camelot)  # ensure the parser picks up the patched base method
tables = camelot.read_pdf('/tmp/camelot-385/1b.pdf', flavor='lattice')
print('after fix [6,1]:', repr(tables[0].df.iat[6, 1]))
# want the 'd' back in place: 'Eprazinon ' then 'd'+'ihydrocl' then 'orid'
# (a newline between 'd' and 'ihydrocl' is acceptable -- playa split that line)

## Step 3 — regression sweep (the real verdict)

The monkey-patch only affects this kernel. For the true verdict, apply the same change to `camelot/parsers/base.py` `compute_parse_errors` on disk (replace the `for direction in [...]:` double-loop with the sorted single loop above), then run the full lattice + stream + network + hybrid suites:

```bash
!python -m pytest tests/test_lattice.py tests/test_stream.py tests/test_network.py tests/test_hybrid.py -q
```

**Decision:** if everything passes, the fix is shippable as-is. If a fixture shifts, eyeball whether the new order is *more* correct (then update the fixture) or whether the global sort disturbed a rotated-text case (then restrict the reorder to same-row textlines only).

In [ ]:
# Convenience: apply the patch to base.py ON DISK, then run the suites.
# (Uncomment to run -- edits the file in the checkout.)
#
# import subprocess
# r = subprocess.run(['python','-m','pytest','tests/test_lattice.py',
#                     'tests/test_stream.py','tests/test_network.py',
#                     'tests/test_hybrid.py','-q'], capture_output=True, text=True)
# print(r.stdout[-4000:])